# Model Extraction: Stealing a Credit-Card Fraud Classifier

This notebook tells the story of a **model extraction (model stealing)** attack. A bank's fraud detector (an RBF SVM) sits behind an API that only returns a hard label - `fraud` or `legit`. With nothing but query access, we rebuild a copy of it.

All attack logic is imported from `train_victim.py` and `extract.py` - this notebook adds **no duplicate logic**, only exploration and visualization.

**Dataset:** Kaggle Credit Card Fraud Detection (284,807 transactions, ~0.17% fraud)  
**Victim:** `StandardScaler` + `SVC(kernel='rbf', probability=False, class_weight='balanced')`  
**Attack:** query the victim 10,000x, train 3 shadow models, keep the best clone.

> Run `python train_victim.py` and `python extract.py` once before this notebook so the saved artifacts exist.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# The lab scripts live in the parent folder; add it to the path so we can
# IMPORT the victim trainer and the attack logic instead of re-writing it.
LAB_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(LAB_DIR))

import train_victim as victim_mod   # load_data()
import extract as atk               # the whole attack toolkit

sns.set_theme(style='whitegrid')
print('Imports OK. Lab dir:', LAB_DIR)

## 1. Load the data and see the class imbalance

Fraud is *rare*. That rarity is the single most important fact about this dataset - it shapes every modelling choice (hence `class_weight='balanced'` everywhere).

In [ ]:
# Use the SAME loader the victim uses, so we see exactly what it sees.
X, y = victim_mod.load_data()

counts = y.value_counts().sort_index()
n_legit, n_fraud = int(counts.get(0, 0)), int(counts.get(1, 0))
fraud_pct = 100 * n_fraud / len(y)
print(f'Legit: {n_legit:,}   Fraud: {n_fraud:,}   Fraud share: {fraud_pct:.3f}%')

labels = ['Legit (0)', 'Fraud (1)']
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
# Left: linear scale -- the fraud bar is basically invisible (that's the point).
axes[0].bar(labels, counts.values, color=['#4c72b0', '#c44e52'])
axes[0].set_title('Class counts (linear scale)')
axes[0].set_ylabel('transactions')
# Right: log scale -- now you can actually SEE how tiny the fraud class is.
axes[1].bar(labels, counts.values, color=['#4c72b0', '#c44e52'])
axes[1].set_yscale('log')
axes[1].set_title('Class counts (log scale)')
axes[1].set_ylabel('transactions (log)')
plt.tight_layout(); plt.show()

## 2. How good is the victim?

We load the trained victim and score it on the shared held-out test set. On imbalanced data, **accuracy lies** - read the precision/recall for the fraud row and the confusion matrix instead.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Load the trained victim and the shared held-out test set.
victim = atk.load_victim()
X_test, y_test = atk.load_test()

victim_preds = victim.predict(X_test)
print(classification_report(y_test, victim_preds,
      target_names=['Legit (0)', 'Fraud (1)'], digits=3))

cm = confusion_matrix(y_test, victim_preds)
plt.figure(figsize=(4.6, 3.9))
sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues',
            xticklabels=['pred legit', 'pred fraud'],
            yticklabels=['actual legit', 'actual fraud'])
plt.title('Victim SVM -- confusion matrix (test set)')
plt.tight_layout(); plt.show()

## 3. The extraction process: more queries -> better clone

For a range of **query budgets** we: draw that many unlabeled transactions, ask the victim to label them, train each shadow on the stolen `(features, victim-label)` pairs, and measure **agreement** with the victim on the test set.

Agreement = the fraction of test transactions where the shadow predicts the *same* label as the victim. It is the real measure of how well we cloned the model.

In [ ]:
# Reference labels = what the victim itself predicts on the test set.
pool = atk.load_pool()
victim_test_preds = victim.predict(X_test)

budgets = [100, 250, 500, 1000, 2000, 5000, 10000]
curves = {name: [] for name in atk.get_shadow_models().keys()}

for n in budgets:
    # Steal n labels by querying the victim (same seed -> reproducible).
    X_stolen, y_stolen = atk.build_stolen_dataset(victim, pool, n)

    # Guard: fraud is so rare that small budgets can come back ALL legit.
    # A single-class stolen set can't train a classifier (LogReg raises
    # ValueError), so we record a gap (None) for this budget and move on.
    if len(np.unique(y_stolen)) < 2:
        for name in curves:
            curves[name].append(None)
        print(f'  budget {n:>6,} skipped (only one class in stolen labels)')
        continue

    # Train a FRESH copy of each shadow on this budget's stolen data.
    for name, model in atk.get_shadow_models().items():
        atk.train_shadow(model, X_stolen, y_stolen)
        agree = atk.agreement_rate(victim_test_preds, model.predict(X_test))
        curves[name].append(agree)
    print(f'  budget {n:>6,} done')

plt.figure(figsize=(8, 5))
for name, ys in curves.items():
    # Drop the skipped budgets (None) so the line only spans budgets we scored.
    valid = [(b, a) for b, a in zip(budgets, ys) if a is not None]
    if valid:
        xs, as_ = zip(*valid)
        plt.plot(xs, [a * 100 for a in as_], marker='o', label=name)
plt.xscale('log')
plt.xlabel('Number of victim queries (log scale)')
plt.ylabel('Agreement with victim (%)')
plt.title('Extraction: more queries -> better clone')
plt.legend(); plt.tight_layout(); plt.show()

## 4. Final comparison: victim vs. all 3 shadow models

At the full 10,000-query budget, how do the three clones stack up? The victim agrees with itself 100% by definition; the winning shadow is the one whose agreement comes closest.

In [ ]:
# Full-budget extraction for the final table.
X_stolen, y_stolen = atk.build_stolen_dataset(victim, pool, atk.N_QUERIES)

rows = []
victim_acc = (np.asarray(y_test) == victim_test_preds).mean()
rows.append({'Model': 'Victim (SVM)', 'Type': 'victim',
             'Accuracy %': round(victim_acc * 100, 2), 'Agreement %': 100.00})

for name, model in atk.get_shadow_models().items():
    atk.train_shadow(model, X_stolen, y_stolen)
    preds = model.predict(X_test)
    acc = (np.asarray(y_test) == preds).mean()
    agree = atk.agreement_rate(victim_test_preds, preds)
    rows.append({'Model': name, 'Type': 'shadow',
                 'Accuracy %': round(acc * 100, 2),
                 'Agreement %': round(agree * 100, 2)})

comparison = pd.DataFrame(rows)
best = (comparison[comparison.Type == 'shadow']
        .sort_values('Agreement %', ascending=False).iloc[0])
print(f"Best clone: {best.Model}  ({best['Agreement %']}% agreement with victim)")
comparison

## 5. Key takeaway

**What just happened:** using only black-box, *hard-label* queries - no access to the victim's data, weights, or code - we trained models that reproduce the victim's decisions on the large majority of test transactions. The agreement curve shows the clone gets steadily better as we spend more queries.

**Why it matters:**
- **Stolen IP:** the bank paid to build its model; we copied its behaviour for the price of some API calls.
- **Stepping stone:** a local clone lets an attacker design *evasion* attacks (Lab 1) offline, then replay them against the live API.
- **Hard labels are not safe:** we never needed confidence scores. Returning probabilities would only make stealing easier.

**Defenses:** query rate-limiting and quotas, monitoring for boundary-probing query patterns, output perturbation/coarsening, and watermarking to prove a clone is a copy.

_See the README's 'Why This Attack Matters in the Real World' section for more._